# Laboratorio #3

* Josue Say - 228801
* Flavio Galán - 22386

## Repositorio

- [Enlace](https://github.com/JosueSay/labs-ds/tree/main/lab4)

## Librerías y constantes

In [ ]:
import csv
import rasterio
import numpy as np
import os, json, time, warnings

from pathlib import Path
from datetime import datetime, timedelta

import openeo
import geopandas as gpd
from shapely.geometry import mapping

from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS as RIO_CRS
from dotenv import load_dotenv
from sentinelhub import (
    SHConfig, SentinelHubRequest, DataCollection, MimeType,
    BBox, bbox_to_dimensions, CRS
)
from sentinelhub.exceptions import SHRateLimitWarning, DownloadFailedException

warnings.filterwarnings("ignore", category=SHRateLimitWarning)

# !pip install -r requirements.txt

In [ ]:
# === GLOBAL ===
BASE_DIR = Path.cwd()
DATA_NAME = "data"
DATA_DIR = BASE_DIR / DATA_NAME

# === SECTION 1 ===
# Fechas con nubosidad <20%
PROF_DATES = [
    "2025-02-07","2025-02-10","2025-02-25","2025-02-27",
    "2025-03-02","2025-03-04","2025-03-07","2025-03-09","2025-03-12","2025-03-14","2025-03-19","2025-03-22","2025-03-24","2025-03-26",
    "2025-04-03","2025-04-11","2025-04-13","2025-04-15","2025-04-16","2025-04-18","2025-04-28",
    "2025-05-03","2025-05-13","2025-05-28",
    "2025-07-10","2025-07-17","2025-07-20","2025-07-24","2025-08-01",
]

USE_SUBSET = False        # ¿Usar un subconjunto (2 fechas por mes) en vez de todas?
SUBSET_MONTHS = [6, 7]    # junio, julio
PER_MONTH = 2             # 2 fechas por mes

# === SECTION 2 ===
MAX_CLOUD = 20
RESOLUTION = 20  # metros
DO_RGB_PREVIEW = False  # True para un GTIFF RGB por lago

BANDS_RGB   = ["B04","B03","B02"]
BANDS_INDEX = ["B02","B03","B04","B05","B08","B8A","B11","B12"]  # para NDVI/NDWI/Cyano

# === SECTION 3 ===
OUT_SUBDIR  = "Cyano_SH"

# Ritmo de descarga
PAUSE_BETWEEN_REQUESTS = 2.0   # seg, pausa fija entre fechas
MAX_RETRIES            = 5     # reintentos ante 429/limit
BASE_BACKOFF_SEC       = 5.0   # backoff exponencial: 5, 10, 20, ...

EVALSCRIPT_CHL = """
function setup() {
  return { input:["B02","B03","B04","B05","B07","B08","B8A","B11","B12"],
           output:{ bands:1, sampleType:"FLOAT32"} };
}
var MNDWI_threshold=0.42, NDWI_threshold=0.4, filter_UABS=true;
function evaluatePixel(s){
  let r=s.B04,g=s.B03,b=s.B02,nir=s.B08,b05=s.B05,b8a=s.B8A,sw1=s.B11,sw2=s.B12;
  let ndvi=(nir-r)/(nir+r), mndwi=(g-sw1)/(g+sw1), ndwi=(g-nir)/(g+nir),
      ndwi_leaves=(nir-sw1)/(nir+sw1),
      aweish=b+2.5*g-1.5*(nir+sw1)-0.25*sw2,
      aweinsh=4*(g-sw1)-(0.25*nir+2.75*sw1),
      dbsi=(sw1-g)/(sw1+g)-ndvi;
  let water=0;
  if(mndwi>MNDWI_threshold||ndwi>NDWI_threshold||aweinsh>0.1879||aweish>0.1112||ndvi<-0.2||ndwi_leaves>1){water=1;}
  if(filter_UABS && water===1){ if(aweinsh<=-0.03||dbsi>0){water=0;} }
  let ndci=(b05-r)/(b05+r);
  let chl=826.57*Math.pow(ndci,3)-176.43*Math.pow(ndci,2)+19*ndci+4.071;
  if(water===0||!isFinite(chl)){return [0.0];} else {return [chl];}
}
"""

# === SECTION 4 ===
DIRS = {
    ("Atitlan", "NDVI"): DATA_DIR / "NDVI_Atitlan",
    ("Amatitlan", "NDVI"): DATA_DIR / "NDVI_Amatitlan",
    ("Atitlan", "NDWI"): DATA_DIR / "NDWI_Atitlan",
    ("Amatitlan", "NDWI"): DATA_DIR / "NDWI_Amatitlan",
    
    ("Atitlan", "CYANO"): DATA_DIR / "Atitlan" / "Cyano_SH",
    ("Amatitlan", "CYANO"): DATA_DIR / "Amatitlan" / "Cyano_SH",
}

OUT_DIR = DATA_DIR / "export_consolidated"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INTERVALS_PATH = DATA_DIR / "intervals.json"

## Fechas y nubosidad (inciso 3 -> p1)

**Objetivo:**

* Definir el rango que cubra el período de estudio (p.ej., feb–ago 2025).
* Usar las fechas de la profesora (nubosidad <20%).
* Armar “ventanas” de 1 día para mosaicos diarios (o 2 fechas/mes si así lo pide).

**Por qué:**

* Garantiza calidad (menos nubes) y uniformidad temporal para el análisis.

**Resultado:**

* Un conjunto de **fechas objetivo** sobre las que harás mosaicos/estadísticos.

In [ ]:
def buildDailyIntervals(date_list):
    """Devuelve [[startZ, endZ], ...] con ventanas de 1 día."""
    intervals = []
    for d in date_list:
        dt = datetime.fromisoformat(d)
        start = dt.isoformat() + "Z"
        end = (dt + timedelta(days=1)).isoformat() + "Z"
        intervals.append([start, end])
    return intervals

In [ ]:
def pickDatesPerMonth(date_list, months, per_month=2):
    """
    Toma 'per_month' fechas por cada mes de 'months' manteniendo orden.
    months: lista de enteros (1-12)
    """
    picked = []
    counters = {m: 0 for m in months}
    for d in date_list:
        m = datetime.fromisoformat(d).month
        if m in counters and counters[m] < per_month:
            picked.append(d)
            counters[m] += 1
    return picked

In [ ]:
def saveDatesAndIntervals(out_dir: Path, dates, intervals):
    """Guarda las fechas en 'dates.txt' y los intervalos en 'intervals.json' dentro del directorio especificado."""
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "dates.txt").write_text("\n".join(dates), encoding="utf-8")
    with (out_dir / "intervals.json").open("w", encoding="utf-8") as f:
        json.dump({"intervals": intervals}, f, indent=2, ensure_ascii=False)

In [ ]:
def executeSection1():
    """Genera intervalos diarios a partir de las fechas configuradas y guarda los resultados en archivos de salida."""

    if USE_SUBSET:
        dates = pickDatesPerMonth(PROF_DATES, SUBSET_MONTHS, PER_MONTH)
    else:
        dates = PROF_DATES

    intervals = buildDailyIntervals(dates)
    saveDatesAndIntervals(DATA_DIR, dates, intervals)

    print(f"[OK] Fechas: {len(dates)} -> guardadas en {DATA_DIR/'dates.txt'}")
    print(f"[OK] Intervalos: {len(intervals)} -> guardados en {DATA_DIR/'intervals.json'}")
    print("Ejemplo primer intervalo:", intervals[0] if intervals else "N/A")


## Conexión, AOI y bandas (incisos 1, 2 y parte de 3)

**Objetivo:**

* Crear conexión a **openEO/CDSE**.
* Cargar el **GeoJSON** de cada lago y obtener su **geometría** (no bbox) para recortar en el servidor.
* Cargar **SENTINEL2\_L2A** con filtro de nubosidad (*server-side*).
* Pedir solo las **bandas necesarias** según el índice:

  * **RGB:** B04, B03, B02 (vista real).
  * **NDVI/NDWI:** B08 (NIR), B04 (Red), B03 (Green).
  * **Cianobacteria (NDCI + máscara agua):** B02, B03, B04, B05, B08, B8A, B11, B12.

**Por qué:**

* Recortar en el servidor evita descargar zonas irrelevantes.
* Menos bandas = menos datos y mayor velocidad; pero para el índice de cianobacteria se necesitan todas las que indica el script.

**Resultado:**

* Obtener un *DataCube* por lago, con geometría aplicada y solo las bandas/fechas necesarias, ya **recortado** y opcionalmente **re-muestreado** (p. ej. 20 m) para acelerar el procesamiento.

In [ ]:
def connectToSentinelHub():
    """Conecta con el endpoint de openEO/CDSE y autentica mediante OIDC."""
    return openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()


In [ ]:
def readLakeGeometry(geojson_path: str):
    """Lee un GeoJSON y devuelve la geometría unificada en formato dict (EPSG:4326)."""
    gdf = gpd.read_file(geojson_path)
    if gdf.crs is None:
        gdf.set_crs(4326, inplace=True)
    else:
        gdf = gdf.to_crs(4326)

    geom = gdf.geometry.union_all()

    return mapping(geom)   # dict con {"type": "...", "coordinates": ...}


In [ ]:
def loadS2BaseCube(conn, geometry, start_date, end_date, bands):
    """Carga un DataCube Sentinel-2 L2A filtrado por fecha, AOI, bandas y nubosidad, 
    lo escala a reflectancia y lo remuestrea a la resolución indicada."""
    cube = conn.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=geometry,
        temporal_extent=[start_date, end_date],
        bands=bands,
        max_cloud_cover=MAX_CLOUD
    )
    cube = cube * 0.0001
    cube = cube.resample_spatial(resolution=RESOLUTION)
    return cube


In [ ]:
def aggregateByIntervals(cube, intervals):
    """Agrega el DataCube en intervalos temporales dados usando la mediana como reductor."""
    return cube.aggregate_temporal(intervals=intervals, reducer="median") # intervals = [[startZ, endZ], ...]


In [ ]:
def buildNdviCube(cube):
    """Calcula el índice NDVI (vegetación) a partir del DataCube Sentinel-2."""
    nir = cube.band("B08"); red = cube.band("B04")
    return (nir - red) / (nir + red)

def buildNdwiCube(cube):
    """Calcula el índice NDWI (agua) a partir del DataCube Sentinel-2."""
    green = cube.band("B03"); nir = cube.band("B08")
    return (green - nir) / (green + nir)

def buildCyanoChlaCube(cube):
    """Calcula la concentración de clorofila-a cianobacteriana usando múltiples índices y máscaras de agua."""
    blue  = cube.band("B02"); green = cube.band("B03"); red = cube.band("B04")
    b05   = cube.band("B05"); nir   = cube.band("B08"); b8a  = cube.band("B8A")
    swir1 = cube.band("B11"); swir2 = cube.band("B12")

    ndvi = (nir - red) / (nir + red)
    mndwi = (green - swir1) / (green + swir1)
    ndwi  = (green - nir)   / (green + nir)
    ndwi_leaves = (nir - swir1) / (nir + swir1)
    aweish  = blue + 2.5*green - 1.5*(nir + swir1) - 0.25*swir2
    aweinsh = 4*(green - swir1) - (0.25*nir + 2.75*swir1)
    dbsi = ((swir1 - green) / (swir1 + green)) - ndvi

    water = (mndwi > 0.42) | (ndwi > 0.4) | (aweinsh > 0.1879) | (aweish > 0.1112) | (ndvi < -0.2) | (ndwi_leaves > 1)
    water = water & ~((aweinsh <= -0.03) | (dbsi > 0))

    ndci = (b05 - red) / (b05 + red)
    chl  = 826.57 * (ndci ** 3) - 176.43 * (ndci ** 2) + 19 * ndci + 4.071
    return chl * water

In [ ]:
def downloadCubeAsTiff(conn, cube, out_dir: Path):
    """Descarga un DataCube como archivos GeoTIFF en el directorio especificado."""
    out_dir.mkdir(parents=True, exist_ok=True)
    result = cube.save_result(format="GTIFF")
    job = conn.create_job(result)
    job.start_and_wait()
    job.get_results().download_files(str(out_dir))


In [ ]:
def executeSection2():
    """Carga, procesa y descarga índices satelitales para los lagos Atitlán y Amatitlán."""

    # Fechas/intervalos del p2.py
    intervals_path = DATA_DIR / "intervals.json"
    dates_path = DATA_DIR / "dates.txt"
    assert intervals_path.exists(), "Falta p_data/intervals.json (ejecuta p2.py)"
    with intervals_path.open("r", encoding="utf-8") as f:
        INTERVALS = json.load(f)["intervals"]

    # AOI
    atitlan_geom   = readLakeGeometry(str(DATA_DIR / "Lago_Atitlan.geojson"))
    amatitlan_geom = readLakeGeometry(str(DATA_DIR / "Lago_Amatitlan.geojson"))

    # Rango amplio (cubrir todas las fechas del archivo)
    start_date = INTERVALS[0][0][:10]
    end_date   = INTERVALS[-1][1][:10]

    conn = connectToSentinelHub()

    # --- Cargas por lago para índices ---
    atitlan_base   = loadS2BaseCube(conn, atitlan_geom,   start_date, end_date, BANDS_INDEX)
    amatitlan_base = loadS2BaseCube(conn, amatitlan_geom, start_date, end_date, BANDS_INDEX)

    # Agregación por las fechas exactas (mosaico diario)
    atitlan_daily   = aggregateByIntervals(atitlan_base,   INTERVALS)
    amatitlan_daily = aggregateByIntervals(amatitlan_base, INTERVALS)

    # Índices en la nube
    cy_atitlan     = buildCyanoChlaCube(atitlan_daily)
    cy_amatitlan   = buildCyanoChlaCube(amatitlan_daily)
    ndvi_atitlan   = buildNdviCube(atitlan_daily)
    ndvi_amatitlan = buildNdviCube(amatitlan_daily)
    ndwi_atitlan   = buildNdwiCube(atitlan_daily)
    ndwi_amatitlan = buildNdwiCube(amatitlan_daily)

    # Descargas
    downloadCubeAsTiff(conn, cy_atitlan,      DATA_DIR / "Cyano_Atitlan")
    downloadCubeAsTiff(conn, cy_amatitlan,    DATA_DIR / "Cyano_Amatitlan")
    downloadCubeAsTiff(conn, ndvi_atitlan,    DATA_DIR / "NDVI_Atitlan")
    downloadCubeAsTiff(conn, ndvi_amatitlan,  DATA_DIR / "NDVI_Amatitlan")
    downloadCubeAsTiff(conn, ndwi_atitlan,    DATA_DIR / "NDWI_Atitlan")
    downloadCubeAsTiff(conn, ndwi_amatitlan,  DATA_DIR / "NDWI_Amatitlan")

    # (Opcional) preview RGB
    if DO_RGB_PREVIEW:
        atitlan_rgb_base   = loadS2BaseCube(conn, atitlan_geom,   start_date, end_date, BANDS_RGB)
        amatitlan_rgb_base = loadS2BaseCube(conn, amatitlan_geom, start_date, end_date, BANDS_RGB)
        atitlan_rgb_daily   = aggregateByIntervals(atitlan_rgb_base,   INTERVALS)
        amatitlan_rgb_daily = aggregateByIntervals(amatitlan_rgb_base, INTERVALS)
        downloadCubeAsTiff(conn, atitlan_rgb_daily,   DATA_DIR / "RGB_Atitlan")
        downloadCubeAsTiff(conn, amatitlan_rgb_daily, DATA_DIR / "RGB_Amatitlan")

    print("[OK] p3 terminado. GeoTIFFs en p_data/*")

## Aplicar el script de cianobacteria en la nube (Sentinel Hub Process API)

**Objetivo:**

* Leer la **AOI (Área de Interés)** desde archivos **GeoJSON** ubicados en `p_data`.
* Cargar los intervalos de fechas desde `intervals.json` generado por `p2.py`.
* Ejecutar un **evalscript** que calcula el índice numérico "Cyanobacteria Chlorophyll-a NDCI" para detectar clorofila.
* Descargar un archivo **TIFF** por cada fecha e intervalo para cada lago, con la clorofila en una sola banda, en formato **FLOAT32**.

**Por qué:**

* Automatizar la descarga de productos derivados de Sentinel-2 para monitoreo de cianobacterias.
* Usar la API de Sentinel Hub permite procesamiento en la nube, evitando descarga y manejo de imágenes brutas grandes.
* Descargar sólo las fechas e intervalos relevantes, reduciendo almacenamiento y tiempos.

**Resultado:**

* Archivos TIFF con mapas de clorofila, organizados por lago y fecha en la carpeta `p_data/<Lago>/Cyano_SH/`.
* Cada TIFF contiene una banda con valores de clorofila calculados a partir del índice NDCI y una máscara de agua, listos para análisis posteriores.

In [ ]:
def getConfig():
    """
    Obtiene la configuración de SHConfig con valores de variables de entorno.
    Valida que CLIENT_ID y CLIENT_SECRET estén definidos.
    """
    cfg = SHConfig()
    cfg.sh_client_id     = os.getenv("CLIENT_ID", cfg.sh_client_id)
    cfg.sh_client_secret = os.getenv("CLIENT_SECRET", cfg.sh_client_secret)
    if os.getenv("SH_TOKEN_URL"):
        cfg.sh_token_url = os.environ["SH_TOKEN_URL"]
    assert cfg.sh_client_id and cfg.sh_client_secret, "Configura CLIENT_ID/CLIENT_SECRET en .env"
    return cfg

In [ ]:
def readLakeGeometry(path_geojson: Path):
    """
    Lee un archivo GeoJSON y devuelve la geometría combinada en CRS 4326.
    """
    gdf = gpd.read_file(path_geojson)
    gdf = gdf.set_crs(4326) if gdf.crs is None else gdf.to_crs(4326)
    return gdf.geometry.union_all()

In [ ]:
def requestForInterval(geom_shp, time_start, time_end, evalscript, cfg, out_dir: Path, target_filename: str):
    """
    Realiza una solicitud a Sentinel Hub para un intervalo de tiempo y guarda el resultado TIFF.
    """
    # BBox WGS84
    minx, miny, maxx, maxy = geom_shp.bounds
    bbox_wgs84 = BBox((minx, miny, maxx, maxy), crs=CRS.WGS84)
    # Dimensiones a ~RESOLUTION m/px (calcular en 3857)
    bbox_3857 = bbox_wgs84.transform(CRS.POP_WEB)
    width, height = bbox_to_dimensions(bbox_3857, RESOLUTION)

    req = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(time_start, time_end),
                other_args={"dataFilter": {"maxCloudCoverage": MAX_CLOUD}}
            )
        ],
        responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
        bbox=bbox_wgs84,
        size=(width, height),
        config=cfg,
    )

    # Descarga con reintentos/backoff ante rate limit
    attempt = 0
    success = False
    data = None

    while not success and attempt <= MAX_RETRIES:
        try:
            data = req.get_data(save_data=False)
            success = True
        except DownloadFailedException as e:
            msg = str(e).lower()
            if attempt < MAX_RETRIES and ("rate limit" in msg or "429" in msg or "too many requests" in msg):
                wait = BASE_BACKOFF_SEC * (2 ** attempt)
                time.sleep(wait)
                attempt += 1
            else:
                raise

    if not success or not data or data[0] is None:
        raise RuntimeError(f"Sin datos para {time_start}–{time_end}")

    arr = data[0]
    if arr.ndim == 3 and arr.shape[2] == 1:
        arr = arr[:, :, 0]
    arr = arr.astype(np.float32)

    out_dir.mkdir(parents=True, exist_ok=True)
    dest = out_dir / target_filename
    transform = from_bounds(minx, miny, maxx, maxy, width, height)
    profile = {
        "driver": "GTiff",
        "height": arr.shape[0],
        "width": arr.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": RIO_CRS.from_epsg(4326),
        "transform": transform,
        "compress": "deflate",
        "predictor": 2,
        "tiled": True
    }
    with rasterio.open(dest, "w", **profile) as dst:
        dst.write(arr, 1)

    return dest

In [ ]:
def processLake(name: str, geojson_path: Path, intervals, out_root: Path, cfg):
    """
    Procesa un lago descargando datos para intervalos de tiempo y guardándolos en archivos TIFF.
    """
    assert geojson_path.exists(), f"No existe: {geojson_path}"
    geom = readLakeGeometry(geojson_path)
    lake_out = out_root / name / OUT_SUBDIR
    lake_out.mkdir(parents=True, exist_ok=True)

    for i, (startZ, endZ) in enumerate(intervals, 1):
        date_label = startZ[:10]
        print(f"[{name}] {i}/{len(intervals)} -> {date_label}")
        requestForInterval(geom, startZ, endZ, EVALSCRIPT_CHL, cfg, lake_out, f"{name}_chl_{date_label}.tif")
        time.sleep(PAUSE_BETWEEN_REQUESTS)  # pausa fija entre requests

In [ ]:
def executeSection3():
    """Descarga datos de clorofila para lagos específicos en intervalos definidos y guarda los archivos TIFF."""
    intervals = json.loads((DATA_DIR / "intervals.json").read_text(encoding="utf-8"))["intervals"]
    cfg = getConfig()

    processLake("Atitlan",   DATA_DIR / "Lago_Atitlan.geojson",   intervals, DATA_DIR, cfg)
    processLake("Amatitlan", DATA_DIR / "Lago_Amatitlan.geojson", intervals, DATA_DIR, cfg)

    print("[OK] p4 completado. TIFFs en p_data/<Lago>/Cyano_SH/*.tif")

## Exportación consolidada

**Objetivo:**

* Convertir los archivos GeoTIFF individuales descargados en GeoTIFFs multitemporales apilados (1 banda = 1 fecha).
* Inventariar las bandas con fechas para facilitar análisis posteriores.
* Generar archivos livianos y organizados para cada índice y lago.

**Por qué:**

* Reduce la cantidad de archivos al apilar fechas en un solo archivo.
* Facilita la gestión y análisis de los datos multitemporales.
* Facilita la revisión rápida con inventarios CSV.

**Resultado:**

* Carpetas `export_consolidated/` con GeoTIFFs multitemporales para NDVI, NDWI y CYANO por lago.
* Archivos CSV con inventarios de bandas y fechas para cada archivo TIFF multitemporal.

In [ ]:
def readIntervals():
    """Lee y devuelve la lista de intervalos de fechas desde el archivo intervals.json."""
    with INTERVALS_PATH.open("r", encoding="utf-8") as f:
        return json.load(f)["intervals"]  # [[startZ,endZ], ...]

In [ ]:
def listTiffs(folder: Path):
    """Lista y ordena todos los archivos .tif dentro del directorio dado."""
    return sorted([p for p in folder.glob("*.tif")])

In [ ]:
def isSingleMultitemporal(tif_path: Path) -> bool:
    """Determina si el path es un único GeoTIFF multibanda (multitemporal)."""
    tiffs = listTiffs(tif_path) if tif_path.is_dir() else [tif_path]
    if len(tiffs) != 1:
        return False
    with rasterio.open(tiffs[0]) as src:
        return src.count > 1  # más de una banda

In [ ]:
def stackManySinglesToMultitemporal(folder: Path, out_path: Path, dates: list):
    """
    Apila muchos TIFF (uno por fecha, 1 banda) -> 1 multibanda,
    REPROYECTANDO cada uno a la grilla del primero.
    """
    tiffs = listTiffs(folder)
    if not tiffs:
        raise RuntimeError(f"No .tif files in {folder}")

    # Orden por fecha en nombre
    def dateKey(p: Path):
        s = p.stem
        for d in dates:
            if d in s:
                return d
        return "9999-12-31"
    tiffs_sorted = sorted(tiffs, key=dateKey)

    # Grilla de referencia = primer TIFF
    with rasterio.open(tiffs_sorted[0]) as ref:
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_h, ref_w = ref.height, ref.width
        dtype = ref.dtypes[0]
        profile = ref.profile.copy()
        profile.update(count=len(tiffs_sorted), compress="deflate", predictor=2, tiled=True, nodata=np.nan)

    arrays = []
    for tif in tiffs_sorted:
        with rasterio.open(tif) as src:
            if (src.crs == ref_crs and src.transform == ref_transform
                and src.width == ref_w and src.height == ref_h):
                band = src.read(1).astype(np.float32)
            else:
                band = np.full((ref_h, ref_w), np.nan, dtype=np.float32)
                reproject(
                    source=src.read(1),
                    destination=band,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear,
                    dst_nodata=np.nan
                )
        arrays.append(band[np.newaxis, ...])

    stack = np.concatenate(arrays, axis=0)

    with rasterio.open(out_path, "w", **profile) as dst:
        for i in range(stack.shape[0]):
            dst.write(stack[i], i + 1)

    return [t.name for t in tiffs_sorted]

In [ ]:
def buildBandDateInventory(dates: list, written_order_names=None, single_multitemporal_path: Path = None):
    """
    Genera pares (band_index, date, src_name).
    - Si se apiló desde muchos TIFF: usa written_order_names
    - Si ya era multitemporal: usa 'dates' en orden
    """
    rows = []
    if written_order_names:
        for i, name in enumerate(written_order_names, start=1):
            date = next((d for d in dates if d in name), "")
            rows.append((i, date, name))
    elif single_multitemporal_path:
        with rasterio.open(single_multitemporal_path) as src:
            n = src.count
        for i in range(1, n + 1):
            date = dates[i - 1] if i - 1 < len(dates) else ""
            rows.append((i, date, single_multitemporal_path.name))
    return rows

In [ ]:
def processOne(lake: str, index_name: str, folder: Path, dates: list):
    """
    folder puede contener:
      - muchos TIFF de 1 banda (por fecha) -> se apilan a 1 multitemporal
      - 1 TIFF multibanda -> solo inventario
    """
    out_tif = OUT_DIR / f"{lake}_{index_name}_stack.tif"
    inv_csv = OUT_DIR / f"{lake}_{index_name}_bands_dates.csv"

    if not folder.exists():
        print(f"[WARN] No existe {folder} -> saltando {lake}-{index_name}")
        return

    if isSingleMultitemporal(folder):
        single = listTiffs(folder)[0]
        print(f"[INFO] {lake}-{index_name}: ya multitemporal -> {single.name}")
        if single.resolve() != out_tif.resolve():
            out_tif.write_bytes(single.read_bytes())
        rows = buildBandDateInventory(dates, single_multitemporal_path=single)
    else:
        print(f"[INFO] {lake}-{index_name}: apilando {len(listTiffs(folder))} TIFF -> {out_tif.name}")
        written_order_names = stackManySinglesToMultitemporal(folder, out_tif, dates)
        rows = buildBandDateInventory(dates, written_order_names=written_order_names)

    with inv_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["band_index", "date(YYYY-MM-DD)", "source"])
        for row in rows:
            w.writerow(row)

    print(f"[OK] {lake}-{index_name}: {out_tif.name}, inventario -> {inv_csv.name}")

In [ ]:
def executeSection4():
    # fechas limpias (YYYY-MM-DD) a partir de intervals.json
    intervals = readIntervals()
    dates = [iv[0][:10] for iv in intervals]
    for (lake, idx), folder in DIRS.items():
        processOne(lake, idx, folder, dates)
    print(f"[DONE] Exportación consolidada en {OUT_DIR}/")

In [ ]:
# executeSection1()
# executeSection2()
# executeSection3()
# executeSection4()

## Librerías y constantes

In [ ]:
# !pip install openeo geopandas

In [ ]:
import openeo
import geopandas as gpd

## Inciso 1

Establecer conexión con Sentibler Hub (Copernicus)

In [ ]:
def connectToSentinelHub():
    return openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

## Inciso 2

Leer bounding box desde un GeoJSON

In [ ]:
def getBoundingBoxFromGeojson(path):
    gdf = gpd.read_file(path)
    bounds = gdf.total_bounds  # xmin, ymin, xmax, ymax
    return {
        "west": bounds[0],
        "south": bounds[1],
        "east": bounds[2],
        "north": bounds[3]
    }

## Inciso 3

Cargar cubo de Sentinel-2 para el período

In [ ]:
def loadSentinelCube(connection, bbox, startDate, endDate, bands):
    return connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=[startDate, endDate],
        bands=bands
    )

Descargar cubo como TIFF

In [ ]:
def downloadCubeAsTiff(connection, cube, outputPath):
    result = cube.save_result(format="GTIFF")
    job = connection.create_job(result)
    job.start_and_wait()
    results = job.get_results()
    results.download_files(outputPath)

## Inciso 4

Aplicar script de cianobacteria (Evalscript)

In [ ]:
def applyCyanobacteriaEvalscript(connection, bbox, startDate, endDate):
    evalscript = '''
                // CyanoLakes Chlorophyll-a L1C
                // Jeremy Kravitz & Mark Matthews (2020)
                var MNDWI_threshold=0.42;
                var NDWI_threshold=0.4;
                var filter_UABS=true;
                var filter_SSI=false;

                function wbi(r,g,b,nir,swir1,swir2) {
                    let ws=0;
                    try {
                        var ndvi=(nir-r)/(nir+r);
                        var mndwi=(g-swir1)/(g+swir1);
                        var ndwi=(g-nir)/(g+nir);
                        var ndwi_leaves=(nir-swir1)/(nir+swir1);
                        var aweish=b+2.5*g-1.5*(nir+swir1)-0.25*swir2;
                        var aweinsh=4*(g-swir1)-(0.25*nir+2.75*swir1);
                        var dbsi=((swir1-g)/(swir1+g))-ndvi;
                        if (mndwi > MNDWI_threshold || ndwi > NDWI_threshold || aweinsh > 0.1879 || aweish > 0.1112 || ndvi < -0.2 || ndwi_leaves > 1) {
                            ws = 1;
                        }
                        if (filter_UABS && ws==1) {
                            if ((aweinsh<=-0.03)||(dbsi>0)) {ws=0;}
                        }
                    } catch(err) {ws=0;}
                    return ws;
                }

                function setup() {
                    return {
                        input: ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"],
                        output: { bands: 3 }
                    };
                }

                function evaluatePixel(sample) {
                    let water = wbi(sample.B04,sample.B03,sample.B02,sample.B08,sample.B11,sample.B12);
                    function FAI (a,b,c) {return (b-a-(c-a)*(783-665)/(865-665));}
                    let FAIv = FAI(sample.B04,sample.B07,sample.B8A);
                    function NDCI (a,b) {return (b-a)/(b+a);}
                    let NDCIv = NDCI(sample.B04,sample.B05);
                    let chl = 826.57 * Math.pow(NDCIv, 3) - 176.43 * Math.pow(NDCIv, 2) + 19 * NDCIv + 4.071;

                    if (water==0) return [3*sample.B04,3*sample.B03,3*sample.B02];
                    else if (FAIv>0.08) return [233/255,72/255,21/255];
                    else if (chl<0.5) return [0,0,1.0];
                    else if (chl<1) return [0,0,1.0];
                    else if (chl<2.5) return [0,59/255,1];
                    else if (chl<3.5) return [0,98/255,1];
                    else if (chl<5) return [15/255,113/255,141/255];
                    else if (chl<7) return [14/255,141/255,120/255];
                    else if (chl<8) return [13/255,141/255,103/255];
                    else if (chl<10) return [30/255,226/255,28/255];
                    else return [233/255,72/255,21/255];
                }
                '''

    cube = connection.load_collection(
        "SENTINEL2_L1C",
        spatial_extent=bbox,
        temporal_extent=[startDate, endDate]
    ).evalscript(evalscript)

    return cube


Crear y descargar imagen procesada con script

In [ ]:
# Conexión
print("Connecting")
connection = connectToSentinelHub()

# GeoJSON
print("GEOJSON Atitlán")
atitlanBBox = getBoundingBoxFromGeojson("data/Lago_Atitlan.geojson")
print("GEOJSON Amatitlán")
amatitlanBBox = getBoundingBoxFromGeojson("data/Lago_Amatitlan.geojson")

# Fechas
startDate = "2025-04-01"
endDate = "2025-08-01"

# Bandas requeridas
bands = ["B02", "B03", "B04", "B08"]

# Cubos base
print("Atitlan Cube")
atitlanCube = loadSentinelCube(connection, atitlanBBox, startDate, endDate, bands)
print("Amatitlan Cube")
amatitlanCube = loadSentinelCube(connection, amatitlanBBox, startDate, endDate, bands)

# Descargas base
print("Atitlan Cube as Tiff")
downloadCubeAsTiff(connection, atitlanCube, "data/Bandas_Atitlan")

In [ ]:
print("Amatitlan Cube as Tiff")
downloadCubeAsTiff(connection, amatitlanCube, "data/Bandas_Amatitlan")

In [ ]:
# Cubos con detección de cianobacteria
print("Atitlan cyano")
cyanoAtitlanCube = applyCyanobacteriaEvalscript(connection, atitlanBBox, startDate, endDate)

In [ ]:
print("Amatitlan cyano")
cyanoAmatitlanCube = applyCyanobacteriaEvalscript(connection, amatitlanBBox, startDate, endDate)

In [ ]:
# Descargas procesadas
print("Atitlan cyano cube as Tiff")
downloadCubeAsTiff(connection, cyanoAtitlanCube, "data/Cyano_Atitlan")

In [ ]:
print("Amatitlan cyano cube as Tiff")
downloadCubeAsTiff(connection, cyanoAmatitlanCube, "data/Cyano_Amatitlan")

## Inciso 5

### Liberías

In [ ]:
import os
import rasterio
import numpy as np

In [ ]:
def loadTifFilesAsArrays(folderPath):
    tifArrays = []
    dates = []

    for filename in sorted(os.listdir(folderPath)):
        if filename.endswith(".tif"):
            path = os.path.join(folderPath, filename)
            with rasterio.open(path) as src:
                array = src.read()  # (bands, height, width)
                tifArrays.append(array)
                # Extraer fecha del nombre del archivo
                date_str = filename.replace("openEO_", "").replace("Z.tif", "")
                dates.append(date_str)

    return dates, tifArrays


In [ ]:
atitlanDates, atitlanArrays = loadTifFilesAsArrays("data/Bandas_Atitlan")
print("Fechas cargadas:", atitlanDates[:5])
print("Forma de primera imagen:", atitlanArrays[0].shape)

## Inciso 6

## Inciso 7

## Inciso 8

## Inciso 9